# 10_final_evaluate
## Comprehensive Feature Selection Evaluation - All Methods

This notebook evaluates and compares:
- **Baseline**: Raw data with all original features
- **Filter Methods**: Variance, ANOVA, Chi-Squared, Correlation, Mutual Information
- **Wrapper Methods (Raw)**: Sklearn SFS and Seeded SFS on raw features
- **Wrapper Methods (Union)**: Sklearn SFS and Seeded SFS on union features

Shows accuracy metrics and percentage improvement over baseline for all methods combined.

### Setup & Configuration

In [ ]:
import sys
sys.path.insert(0, '/home/andev/Dev/pka/feature_selection/wrapper-w-filter')

from pathlib import Path
from src.modeling.evaluation import ModelEvaluator
from src.config import ProjectPath

### Dataset Configuration

In [ ]:
data_name = "Lung_cancer"
n_features = 25

path = ProjectPath(data_name, n_features)

print(f"Dataset: {data_name}")
print(f"Features: {n_features}")
print(f"Evaluation dir: {path.evaluation_dir}")

### Wrapper Configuration (Auto-detected from wrapper files)

In [ ]:
import re

def discover_wrapper_config(data_name, variant):
    """Discover wrapper model and hyperparameters from file names"""
    wrapper_dir = Path(f'data/processed/{data_name}/04_wrapper/{variant}')
    if not wrapper_dir.exists():
        return {}
    
    csv_files = list(wrapper_dir.glob('**/*.csv'))
    info_dict = {}
    
    for csv_file in sorted(csv_files):
        filename = csv_file.name
        pattern = r'[a-zA-Z0-9_]+_(seededsfsselector|sklearnsfsselector)_([a-z]+)_accuracy_([a-z0-9]+max)(?:_([0-9]+seeds))?(?:_([0-9]+pat))?_([0-9]+cv)_([a-z]+)\.csv'
        match = re.match(pattern, filename)
        
        if match:
            algorithm, model, max_f, seeds_str, patience_str, cv_str, var = match.groups()
            max_features = max_f.replace('max', '')
            num_seeds = int(seeds_str.replace('seeds', '')) if seeds_str else 1
            num_pat = int(patience_str.replace('pat', '')) if patience_str else 0
            num_cv = int(cv_str.replace('cv', ''))
            
            key = f"{algorithm}_{model}"
            info_dict[key] = {
                'algorithm': algorithm,
                'model': model,
                'max_features': max_features,
                'seeds': num_seeds,
                'patience': num_pat,
                'cv': num_cv,
            }
    
    # Select best per algorithm (prefer log, then logisticregression)
    by_algo = {}
    for key, info in info_dict.items():
        algo = info['algorithm']
        if algo not in by_algo:
            by_algo[algo] = []
        by_algo[algo].append((key, info))
    
    selected = {}
    for algo, files in by_algo.items():
        if algo == 'seededsfsselector':
            preferred = [f for f in files if f[1]['model'] == 'log']
        else:
            preferred = [f for f in files if f[1]['model'] == 'logisticregression']
        
        if preferred:
            selected[algo] = preferred[0][1]
        else:
            selected[algo] = files[0][1]
    
    return selected

# Discover config for both variants
raw_config = discover_wrapper_config(data_name, 'raw')
union_config = discover_wrapper_config(data_name, 'union')

print("\n" + "="*80)
print("WRAPPER CONFIGURATION (Auto-detected)")
print("="*80)

if raw_config:
    print(f"\nRAW variant found {len(raw_config)} algorithms:")
    for algo, cfg in raw_config.items():
        print(f"  - {algo}: model={cfg['model']}, cv={cfg['cv']}")
else:
    print("\nWARNING: No wrapper files found for RAW variant")

if union_config:
    print(f"\nUNION variant found {len(union_config)} algorithms:")
    for algo, cfg in union_config.items():
        print(f"  - {algo}: model={cfg['model']}, cv={cfg['cv']}")
else:
    print("\nWARNING: No wrapper files found for UNION variant")

### Initialize Evaluator

In [ ]:
evaluator = ModelEvaluator(data_name, custom_base_dir=path.evaluation_dir, n_features=n_features)

### 1. Evaluate Baseline (All Features)

In [ ]:
print("\n" + "="*80)
print("PHASE 1: BASELINE EVALUATION (All Original Features)")
print("="*80)

# Use CV from raw config (or default to 4)
n_splits = next(iter(raw_config.values()))['cv'] if raw_config else 4
evaluator.evaluate_baseline(str(path.raw_path), n_splits=n_splits)

### 2. Evaluate Filter Methods

In [ ]:
print("\n" + "="*80)
print("PHASE 2: FILTER METHODS EVALUATION")
print("="*80)

evaluator.evaluate_filtered_features(str(path.filter_dir), n_splits=n_splits)

### 3. Evaluate Wrapper Methods - Raw Variant

In [ ]:
print("\n" + "="*80)
print("PHASE 3A: WRAPPER METHODS EVALUATION (Raw Variant)")
print("="*80)

# Sklearn SFS - Raw
if 'sklearnsfsselector' in raw_config:
    cfg = raw_config['sklearnsfsselector']
    suffix = f"{cfg['model']}_accuracy_{cfg['max_features']}max_{cfg['cv']}cv_raw"
    path_sklearn_raw = path.wrapper_file(
        data_variant='raw',
        algorithm_name='sklearnsfsselector',
        suffix=suffix
    )
    print(f"\nSklearn SFS (Raw) path: {path_sklearn_raw}")
    if path_sklearn_raw.exists():
        evaluator.evaluate_custom_file(str(path_sklearn_raw), 'Sklearn_SFS_Raw', n_splits=cfg['cv'])
    else:
        print(f"WARNING: File not found: {path_sklearn_raw}")
else:
    print("WARNING: sklearnsfsselector not found in raw config")

In [ ]:
# Seeded SFS - Raw
if 'seededsfsselector' in raw_config:
    cfg = raw_config['seededsfsselector']
    suffix = f"{cfg['model']}_accuracy_{cfg['max_features']}max_{cfg['seeds']}seeds_{cfg['patience']}pat_{cfg['cv']}cv_raw"
    path_seeded_raw = path.wrapper_file(
        data_variant='raw',
        algorithm_name='seededsfsselector',
        suffix=suffix
    )
    print(f"\nSeeded SFS (Raw) path: {path_seeded_raw}")
    if path_seeded_raw.exists():
        evaluator.evaluate_custom_file(str(path_seeded_raw), 'Seeded_SFS_Raw', n_splits=cfg['cv'])
    else:
        print(f"WARNING: File not found: {path_seeded_raw}")
else:
    print("WARNING: seededsfsselector not found in raw config")

### 4. Evaluate Wrapper Methods - Union Variant

In [ ]:
print("\n" + "="*80)
print("PHASE 3B: WRAPPER METHODS EVALUATION (Union Variant)")
print("="*80)

# Sklearn SFS - Union
if 'sklearnsfsselector' in union_config:
    cfg = union_config['sklearnsfsselector']
    suffix = f"{cfg['model']}_accuracy_{cfg['max_features']}max_{cfg['cv']}cv_union"
    path_sklearn_union = path.wrapper_file(
        data_variant='union',
        algorithm_name='sklearnsfsselector',
        suffix=suffix
    )
    print(f"\nSklearn SFS (Union) path: {path_sklearn_union}")
    if path_sklearn_union.exists():
        evaluator.evaluate_custom_file(str(path_sklearn_union), 'Sklearn_SFS_Union', n_splits=cfg['cv'])
    else:
        print(f"WARNING: File not found: {path_sklearn_union}")
else:
    print("WARNING: sklearnsfsselector not found in union config")

In [ ]:
# Seeded SFS - Union
if 'seededsfsselector' in union_config:
    cfg = union_config['seededsfsselector']
    suffix = f"{cfg['model']}_accuracy_{cfg['max_features']}max_{cfg['seeds']}seeds_{cfg['patience']}pat_{cfg['cv']}cv_union"
    path_seeded_union = path.wrapper_file(
        data_variant='union',
        algorithm_name='seededsfsselector',
        suffix=suffix
    )
    print(f"\nSeeded SFS (Union) path: {path_seeded_union}")
    if path_seeded_union.exists():
        evaluator.evaluate_custom_file(str(path_seeded_union), 'Seeded_SFS_Union', n_splits=cfg['cv'])
    else:
        print(f"WARNING: File not found: {path_seeded_union}")
else:
    print("WARNING: seededsfsselector not found in union config")

### 5. Generate Comprehensive Report & Visualization

In [ ]:
print("\n" + "="*80)
print("PHASE 4: GENERATING COMPREHENSIVE REPORT")
print("="*80)

result_df = evaluator.generate_report_and_plot(
    experiment_prefix=f"final_evaluation_all_methods_{data_name}",
    chart_title=f"Comprehensive Feature Selection Evaluation - All Methods"
)

### 6. Summary Table with Improvement Over Baseline

In [ ]:
if result_df is not None:
    summary = result_df.groupby(['Method', 'Model']).agg({
        'Acc': ['mean', 'std', 'count']
    }).round(4)
    summary.columns = ['Mean_Accuracy', 'Std_Accuracy', 'N_Folds']
    summary = summary.reset_index()
    
    baseline_acc = summary[summary['Method'] == 'None']['Mean_Accuracy'].values
    
    if len(baseline_acc) > 0:
        baseline_mean = baseline_acc[0]
        print(f"\nBASELINE ACCURACY (All Features): {baseline_mean:.4f}")
        print("\n" + "="*100)
        print("IMPROVEMENT OVER BASELINE")
        print("="*100)
        
        summary['Improvement'] = summary['Mean_Accuracy'] - baseline_mean
        summary['Improvement_%'] = (summary['Improvement'] / baseline_mean * 100).round(2)
        summary_sorted = summary.sort_values('Improvement_%', ascending=False)
        
        display_df = summary_sorted[['Method', 'Model', 'Mean_Accuracy', 'Std_Accuracy', 'Improvement', 'Improvement_%']].copy()
        display_df['Mean_Accuracy'] = display_df['Mean_Accuracy'].round(4)
        display_df['Std_Accuracy'] = display_df['Std_Accuracy'].round(4)
        display_df['Improvement'] = display_df['Improvement'].round(4)
        
        print(display_df.to_string(index=False))
        
        summary_path = Path(evaluator.report_dir) / f"final_evaluation_all_methods_summary_{data_name}.csv"
        display_df.to_csv(summary_path, index=False)
        print(f"\nSummary saved to: {summary_path}")

### 7. Key Insights

In [ ]:
if result_df is not None and len(baseline_acc) > 0:
    summary_sorted = summary.sort_values('Improvement_%', ascending=False)
    best_method = summary_sorted.iloc[0]
    worst_method = summary_sorted.iloc[-1]
    
    print("\n" + "="*100)
    print("KEY FINDINGS")
    print("="*100)
    print(f"\n✓ BEST PERFORMER:")
    print(f"  Method: {best_method['Method']} ({best_method['Model']})")
    print(f"  Accuracy: {best_method['Mean_Accuracy']:.4f} (±{best_method['Std_Accuracy']:.4f})")
    print(f"  Improvement over baseline: +{best_method['Improvement_%']:.2f}%")
    
    print(f"\n✗ WORST PERFORMER:")
    print(f"  Method: {worst_method['Method']} ({worst_method['Model']})")
    print(f"  Accuracy: {worst_method['Mean_Accuracy']:.4f} (±{worst_method['Std_Accuracy']:.4f})")
    print(f"  Change vs baseline: {worst_method['Improvement_%']:.2f}%")
    
    improved_count = len(summary_sorted[summary_sorted['Improvement_%'] > 0])
    total_count = len(summary_sorted) - 1
    
    print(f"\n📊 OVERALL STATISTICS:")
    print(f"  Methods evaluated: {len(summary_sorted) - 1}")
    print(f"  Methods improving baseline: {improved_count}/{total_count}")
    print(f"  Average improvement: {summary_sorted[summary_sorted['Method'] != 'None']['Improvement_%'].mean():.2f}%")